In [ ]:
import os
import seaborn as sns
import pandas as pd
import numpy as np
import geopandas as gpd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
# import contextily as ctx
import warnings
warnings.filterwarnings('ignore')
import sys
# sys.path.append("/path/to/sinkhole-risk-china/code")# project code path
sys.path.append("/path/to/sinkhole-risk-china/code")# project code path
from mgtwr.sel_bw import Sel_BW
from mgtwr.model import Model
import multiprocessing as mp
import psutil
from sklearn.metrics import r2_score
from shapely.geometry import Point
from pathlib import Path


In [ ]:
ssp = "hist"  # TODO: 根据需要手动改成 'hist' / 'ssp1' / 'ssp2' / 'ssp3' / 'ssp4' / 'ssp5'
ssp_time = "2020"  # TODO: 根据需要手动改成 '2020' / '2040' / '2060' / '2080' / '2100'

In [ ]:
resolution = "10km"  # TODO: 根据需要把分辨率手动改成 '0.1km' / '1km' / '3km' / '5km' / '10km' / '20km'
home_dir = os.path.expanduser("/path/to/home/")
base_path = os.path.join(home_dir, "PROJECT_DIR", "Papers", "4_NC")
data_base_path = os.path.join(base_path, "data")

path_name = f"Points_China_all_{resolution}"
print(path_name)
# ✅ 输出目录：base_path/outputs/path_name
out_dir = os.path.join(base_path, "outputs", path_name, ssp, ssp_time)
print(f"out_dir:{out_dir}")
os.makedirs(out_dir, exist_ok=True)


预测集

In [ ]:


# pre_data_path = os.path.join(data_base_path, "Extracted_HAVE_future", "Positive_Negative_balanced", "AllFeatures_Positive_Negative_balanced_25366_ssp5_cleaned.csv")
pre_data_path = os.path.join(data_base_path, "Extracted_HAVE_future", f"Points_China_all_{resolution}", f"AllFeatures_Points_China_all_{resolution}_{ssp}_cleaned.csv")
pre_df = pd.read_csv(pre_data_path)
print("Columns (list):")
print(pre_df.columns.tolist())

###################################################################################
############################ 选择所需特征###########################################
###################################################################################
# ---- 静态特征 ----
static_features = [
    "Distance_to_karst",
    "Depth_to_Bedrock",
    "Distance_to_Fault_m",
]

# ---- 动态特征前缀 ----
dynamic_prefixes = ["UrbanFrac", "PopTotal", "ImperviousIndex", "LAI", "Precip", "WTD", "HDS", "Tas", "Huss"]

def pick_dynamic_col(prefix: str, ssp: str, ssp_time: str) -> str:
    # 2020 或 hist：用历史列
    if ssp == "hist" or ssp_time == "2020":
        return f"{prefix}_hist_2000_2010_2020"
    # 2040/2060/2080/2100：用 (t-20, t) 的20年窗口列
    t = int(ssp_time)
    return f"{prefix}_{t-20}_{t}"

pre_features = static_features + [pick_dynamic_col(p, ssp, ssp_time) for p in dynamic_prefixes]

# ---- 先检查缺失列，避免直接 KeyError ----
missing = [c for c in pre_features if c not in pre_df.columns]
if missing:
    print("❌ Missing columns:", missing)
    print("\n✅ Available columns (preview):", pre_df.columns.tolist()[:20], "...")
    raise KeyError("Some required features are not in the CSV columns.")

pre_X = pre_df[pre_features].values
print("✅ pre_features =", pre_features)
print("✅ pre_X shape =", pre_X.shape)
###################################################################################
###################################################################################
###################################################################################

# 若 pre_df 没有 Disaster 列，则创建并填充为 3
if 'Disaster' not in pre_df.columns:
    pre_df['Disaster'] = 3

pre_y = pre_df['Disaster'].values.reshape(-1, 1)  # (n_samples, 1)

print("自变量 pre_X 的形状:", pre_X.shape)
print("因变量 pre_y 的形状:", pre_y.shape)
# 创建地理坐标系
geometry = [Point(lon, lat) for lon, lat in zip(pre_df['Longitude'], pre_df['Latitude'])]
pre_gdf = gpd.GeoDataFrame(pre_df, geometry=geometry, crs="EPSG:4326")

# 转换为投影坐标系（使用等距投影）
pre_gdf = pre_gdf.to_crs("EPSG:3857")  # Web Mercator投影
pre_coords1 = np.array([(pt.x, pt.y) for pt in pre_gdf.geometry])
pre_coords = np.hstack((pre_coords1, np.zeros((pre_coords1.shape[0], 1))))
print("坐标pre_coords的形状:", pre_coords.shape)

print("\n✅ pre_features 对应数据前 5 行预览:")
display(pre_df[pre_features].head())

读取当前ssp与ssp_time的数据

In [ ]:
import pickle
from pathlib import Path
from sklearn.metrics import r2_score

# 结果文件名（保存在当前文件夹）
pkl_path = Path(f"gwr_pre_{path_name}_{ssp}_{ssp_time}_results.pkl")

if pkl_path.exists():
    # 读取暂存
    with pkl_path.open("rb") as f:
        saved = pickle.load(f)
    y_pred_gwr = saved["y_pred_gwr"]
    params = saved["params"]
    r2 = saved.get("r2", None)


读取hist 2020的数据

In [ ]:
pkl_path_hist_2020 = Path(f"gwr_pre_{path_name}_hist_2020_results.pkl")

if pkl_path_hist_2020.exists():
    # 读取暂存
    with pkl_path_hist_2020.open("rb") as f:
        saved_hist_2020 = pickle.load(f)
    y_pred_gwr_hist_2020 = saved_hist_2020["y_pred_gwr"]
    params_hist_2020 = saved_hist_2020["params"]
    r2_hist_2020 = saved_hist_2020.get("r2", None)

使用自然断点法分类

读取边界并分类

In [ ]:
# part2_load_and_classify.py
import numpy as np
import pickle
from pathlib import Path
import mgtwr.gwr_sigmoid_utils as gwr_sigmoid_utils

BOUNDARY_CANDIDATES = [
    Path(base_path) / 'code' / '3_gwr_model_train' / 'national' / 'GWR' / 'class_boundaries.pkl',
    Path(base_path) / 'code' / '3_gwr_model_train' / 'national' / 'class_boundaries.pkl',
    Path('class_boundaries.pkl'),
]

def load_class_boundary_data(candidates):
    fallback = None
    fallback_path = None

    for path in candidates:
        if not path.exists():
            continue

        with path.open('rb') as f:
            boundary_data = pickle.load(f)

        if boundary_data.get('transform_metadata') is not None:
            return boundary_data, path

        if fallback is None:
            fallback = boundary_data
            fallback_path = path

    if fallback is not None:
        return fallback, fallback_path

    searched = ', '.join(str(p) for p in candidates)
    raise FileNotFoundError(f'未找到可用的 class_boundaries.pkl。已检查: {searched}')

def convert_scores_for_saved_boundaries(gwr_results, transform_metadata=None):
    scores = np.asarray(gwr_results, dtype=float).reshape(-1)
    valid = np.isfinite(scores)
    if not valid.any():
        raise ValueError('gwr_results 中没有可用的有限值。')

    if transform_metadata is None:
        return np.clip(scores, 0.0, 1.0), None

    probabilities, transform_metadata = gwr_sigmoid_utils.gwr_scores_to_probabilities(
        scores,
        transform_metadata=transform_metadata,
    )
    return probabilities, transform_metadata

def classify_with_saved_boundaries(probabilities, class_breaks):
    probabilities = np.asarray(probabilities, dtype=float).reshape(-1)
    class_breaks = np.sort(np.asarray(class_breaks, dtype=float).reshape(-1))
    labels = np.full(probabilities.shape, -1, dtype=int)
    valid = np.isfinite(probabilities)
    labels[valid] = np.digitize(probabilities[valid], class_breaks[1:-1], right=True)
    labels = np.clip(labels, 0, max(len(class_breaks) - 2, 0))
    return labels

try:
    boundary_data, boundary_path = load_class_boundary_data(BOUNDARY_CANDIDATES)
    class_breaks = np.asarray(boundary_data['class_breaks'], dtype=float)
    transform_metadata = boundary_data.get('transform_metadata')
    stats = boundary_data['probability_stats']
    actual_n_classes = int(boundary_data.get('n_classes', len(class_breaks) - 1))
    requested_n_classes = int(boundary_data.get('requested_n_classes', actual_n_classes))

    print(f"成功读取类别边界: {boundary_path}")
    for i, break_point in enumerate(class_breaks):
        print(f"类别边界 {i}: {break_point:.4f}")

    print(f"\n保存时的概率统计:")
    print(f"最小值: {stats['min']:.4f}")
    print(f"最大值: {stats['max']:.4f}")
    print(f"均值: {stats['mean']:.4f}")
    print(f"请求类别数: {requested_n_classes}")
    print(f"实际类别数: {actual_n_classes}")
    if transform_metadata is None:
        print("\n警告: 读取到的是旧版边界文件，将退回 clip(0,1) 逻辑。")
    else:
        print(f"\n读取到的转换方式: {transform_metadata['method']}")
        print(f"center: {transform_metadata['center']:.4f}")
        print(f"scale: {transform_metadata['scale']:.4f}")

except FileNotFoundError:
    print("错误: 未找到 class_boundaries.pkl 文件")
    print("请先运行训练阶段代码生成类别边界")
    raise

y_pred = np.asarray(y_pred_gwr, dtype=float).reshape(-1)
y_prob, transform_metadata = convert_scores_for_saved_boundaries(y_pred, transform_metadata)
susceptibility_labels = classify_with_saved_boundaries(y_prob, class_breaks)

print("\n新数据的概率统计:")
print(f"最小值: {y_prob.min():.4f}")
print(f"最大值: {y_prob.max():.4f}")
print(f"均值: {y_prob.mean():.4f}")

unique, counts = np.unique(susceptibility_labels, return_counts=True)
print("\n各类别样本分布:")
level_names = ['极低', '低', '中等', '高', '极高']
english_level_names = ['Extremely low', 'Low', 'Moderate', 'High', 'Extremely high']
class_names = {i: english_level_names[i] for i in range(min(actual_n_classes, len(english_level_names)))}

for label, count in zip(unique, counts):
    chinese_name = level_names[label] if label < len(level_names) else f'等级{label}'
    english_name = class_names.get(label, f'Class_{label}')
    percentage = count / len(y_prob) * 100
    print(f"{chinese_name}({english_name}): {count}个样本 ({percentage:.1f}%)")

pre_df['Susceptibility_Prob'] = y_prob
pre_df['Susceptibility_Class'] = susceptibility_labels
pre_df['Susceptibility_Level'] = pre_df['Susceptibility_Class'].map(class_names).fillna('Unclassified')

print("\n分类完成！可以使用 Susceptibility_Prob 和 Susceptibility_Class 进行后续分析")


绘图

In [ ]:
import os
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from scipy.interpolate import griddata
import rasterio
from rasterio.transform import from_origin
from shapely.geometry import Point
import jenkspy
from matplotlib import font_manager
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib_scalebar.scalebar import ScaleBar
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
from pyproj import Transformer
import matplotlib.patheffects as pe

# 设置Times New Roman字体，字体大小调整为原来的两倍
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 18

COLORS_BY_SSP = {
    "hist": ["#ffffff", "#ddd7da", "#c3b4ba", "#a78f98", "#8b6a76"],
    "ssp1": ["#ffffff", "#d8ddd6", "#bcc6b6", "#9eac97", "#7e9278"],
    "ssp2": ["#ffffff", "#d6dcde", "#b9c4c7", "#9dabb0", "#7f9097"],
    "ssp3": ["#ffffff", "#ddd7d0", "#c5b8ab", "#ac9885", "#927a66"],
    "ssp4": ["#ffffff", "#dddccf", "#c4bda8", "#aaa186", "#8f856a"],
    "ssp5": ["#ffffff", "#e0d2ce", "#c8afa8", "#ae8a81", "#946760"],
}

try:
    colors = COLORS_BY_SSP[ssp]
except KeyError:
    raise ValueError(f"未知的 ssp: {ssp}，可选值：{list(COLORS_BY_SSP.keys())}")

print("ssp =", ssp)
print("colors =", colors)

# 定义类别名称
class_names = {0: 'Lowest', 1: 'Low', 2: 'Middle', 3: 'High', 4: 'Highest'}

import matplotlib.patheffects as pe

#-----------函数：添加比例尺--------------
def add_scalebar(ax, lon0, lat0, length, projection, bar_lw=5, outline_lw=0.5):
    """
    在指定坐标添加比例尺，并给比例尺加黑色边框（含右侧端点边框）
    bar_lw: 比例尺主体线宽（pt）
    outline_lw: 黑色边框厚度（pt）
    """
    # 将经纬度转换为投影坐标
    x0, y0 = projection.transform_point(lon0, lat0, ccrs.PlateCarree())

    # 计算比例尺长度在投影坐标系中的距离
    scale_length = length * 1000  # km -> m

    # 描边效果：外圈黑色 + 正常线
    outline_effect = [
        pe.Stroke(linewidth=bar_lw + 2 * outline_lw, foreground="black"),
        pe.Normal()
    ]

    # 1) 两段比例尺（黑 + 白），带描边（提供上下边框）
    ax.plot([x0, x0 + scale_length/2], [y0, y0],
            color="black", lw=bar_lw, solid_capstyle="butt",
            transform=ax.transData, path_effects=outline_effect, zorder=5, clip_on=False)

    ax.plot([x0 + scale_length/2, x0 + scale_length], [y0, y0],
            color="white", lw=bar_lw, solid_capstyle="butt",
            transform=ax.transData, path_effects=outline_effect, zorder=5, clip_on=False)

    # 2) 计算“比例尺半厚度”对应的数据坐标高度，用于画端点竖线（封口）
    half_thick_pt = (bar_lw / 2.0 + outline_lw)          # 半厚度（pt）
    half_thick_px = ax.figure.dpi * half_thick_pt / 72.0 # pt -> px

    x_disp, y_disp = ax.transData.transform((x0, y0))
    _, y_up = ax.transData.inverted().transform((x_disp, y_disp + half_thick_px))
    _, y_dn = ax.transData.inverted().transform((x_disp, y_disp - half_thick_px))

    # 3) 画端点竖线：左端 + 右端（保证右侧边框出现）
    for x_end in (x0, x0 + scale_length):
        ax.plot([x_end, x_end], [y_dn, y_up],
                color="black", lw=2 * outline_lw, solid_capstyle="butt",
                transform=ax.transData, zorder=6, clip_on=False)

    # （可选）如果你也想让黑白分段处有一条竖线分隔，取消注释下面三行
    # x_mid = x0 + scale_length/2
    # ax.plot([x_mid, x_mid], [y_dn, y_up],
    #         color="black", lw=2 * outline_lw, solid_capstyle="butt",
    #         transform=ax.transData, zorder=6, clip_on=False)

    # 添加文字标注（保持你原来的写法）
    ax.text(x0, y0 + scale_length*0.05, '0',
            horizontalalignment='center', fontfamily='Times New Roman', fontsize=22,
            transform=ax.transData, zorder=7)
    ax.text(x0 + scale_length/2, y0 + scale_length*0.05, '500',
            horizontalalignment='center', fontfamily='Times New Roman', fontsize=22,
            transform=ax.transData, zorder=7)
    ax.text(x0 + scale_length, y0 + scale_length*0.05, f'{length} km',
            horizontalalignment='center', fontfamily='Times New Roman', fontsize=22,
            transform=ax.transData, zorder=7)



def load_boundary_data(base_path):
    """
    加载边界数据：省界（不含港澳台）+ 港澳台省界（单独）+ 世界国界 + 九段线
    """
    # 省级边界：不含港澳台
    province_no_TW_AM_HK_shp = os.path.join(
        base_path, "data", "Administrative_divisions_of_china",
        "no_TW_AM_HK", "china_pro2_no_TW_AM_HK.shp"
    )
    province_no_TW_AM_HK = gpd.read_file(province_no_TW_AM_HK_shp)

    # 港澳台单独
    province_TW_AM_HK_shp = os.path.join(
        base_path, "data", "Administrative_divisions_of_china",
        "TW_AM_HK", "TW_AM_HK.shp"
    )
    province_TW_AM_HK = gpd.read_file(province_TW_AM_HK_shp)

    # 世界国界
    world_shp = os.path.join(base_path, "data", "world_SHP", "World_countries.shp")
    world_boundary = gpd.read_file(world_shp)

    # 九段线
    nine_dash_line_path = os.path.join(base_path, "data", "china_SHP", "nanhai", "九段线.shp")
    nine_dash_line = gpd.read_file(nine_dash_line_path)

    return province_no_TW_AM_HK, province_TW_AM_HK, world_boundary, nine_dash_line


def create_susceptibility_raster(pre_df, boundary_shp, output_path, resolution_km):
    """
    创建地质灾害易发性分布图
    """
    print(f"开始创建易发性分布图，分辨率: {resolution_km}km...")
    
    # 1. 读取边界文件
    print("步骤1: 读取边界文件")
    china_boundary = gpd.read_file(boundary_shp)
    china_boundary = china_boundary.to_crs("EPSG:3857")
    print(f"边界文件坐标系: {china_boundary.crs}")
    
    # 2. 获取边界范围并适当扩展以确保覆盖全中国
    print("步骤2: 计算边界范围")
    bounds = china_boundary.total_bounds
    print(f"原始边界范围: {bounds}")
    
    # 扩展边界以确保覆盖全中国（特别是东部和南部地区）
    # 向东扩展5度经度，向南扩展2度纬度
    expansion_factor_degree = 5  # 扩展5度
    expansion_m = expansion_factor_degree * 111000  # 粗略转换：1度≈111km
    
    bounds[0] -= expansion_m  # 向西扩展
    bounds[1] -= expansion_m  # 向南扩展  
    bounds[2] += expansion_m  # 向东扩展
    bounds[3] += expansion_m  # 向北扩展
    
    print(f"扩展后边界范围: {bounds}")
    
    # 3. 创建网格
    print("步骤3: 创建网格")
    resolution_m = resolution_km * 1000
    
    x_min, y_min, x_max, y_max = bounds
    width = x_max - x_min
    height = y_max - y_min
    
    x_res = int(np.round(width / resolution_m))
    y_res = int(np.round(height / resolution_m))
    
    x_max_adj = x_min + x_res * resolution_m
    y_max_adj = y_min + y_res * resolution_m
    
    x = np.linspace(x_min, x_max_adj, x_res)
    y = np.linspace(y_min, y_max_adj, y_res)
    xx, yy = np.meshgrid(x, y)
    print(f"网格尺寸: {xx.shape}, 分辨率: {resolution_km}km")
    
    # 4. 准备插值数据
    print("步骤4: 准备插值数据")
    geometry = [Point(lon, lat) for lon, lat in zip(pre_df['Longitude'], pre_df['Latitude'])]
    gdf_points = gpd.GeoDataFrame(pre_df, geometry=geometry, crs="EPSG:4326")
    gdf_points = gdf_points.to_crs("EPSG:3857")
    
    points_coords = np.array([(pt.x, pt.y) for pt in gdf_points.geometry])
    susceptibility_values = pre_df['Susceptibility_Class'].values
    
    print(f"插值点数量: {len(points_coords)}")
    
    # 5. 最近邻插值
    print("步骤5: 执行最近邻插值")
    grid_values = griddata(points_coords, susceptibility_values, 
                          (xx, yy), method='nearest', fill_value=-9999)
    
    # 6. 创建边界掩膜
    print("步骤6: 创建边界掩膜")
    grid_points = [Point(x, y) for x, y in zip(xx.ravel(), yy.ravel())]
    grid_gdf = gpd.GeoDataFrame(geometry=grid_points, crs="EPSG:3857")
    
    points_in_china = gpd.sjoin(grid_gdf, china_boundary, how='inner', predicate='within')
    
    mask = np.zeros(xx.shape, dtype=bool)
    valid_indices = points_in_china.index
    mask.ravel()[valid_indices] = True
    
    grid_values[~mask] = -9999
    
    print(f"有效网格点数量: {np.sum(mask)}")
    
    # 7. 保存为TIF文件
    print("步骤7: 保存TIF文件")
    transform = from_origin(x_min, y_max_adj, resolution_m, resolution_m)
    
    with rasterio.open(
        output_path,
        'w',
        driver='GTiff',
        height=grid_values.shape[0],
        width=grid_values.shape[1],
        count=1,
        dtype=grid_values.dtype,
        crs=china_boundary.crs,
        transform=transform,
        nodata=-9999
    ) as dst:
        dst.write(grid_values, 1)
    
    print(f"TIF文件已保存: {output_path}")
    
    return grid_values, xx, yy, (x_min, y_min, x_max_adj, y_max_adj), transform

def create_preview_map(grid_values, bounds, output_base_path, resolution_km, base_path):
    """
    创建符合Nature风格的预览图像，使用兰伯特投影
    - 图更紧凑（宽度变小）
    - 经度范围收窄（左边界+一点，右边界-一点）
    - 省界拆分：不含港澳台用于分析；绘图带港澳台（白色填充+黑边）
    - 图例增加 No data（白色，放第一个），色块黑边框 lw=1
    - 加九段线
    """
    print(f"步骤8: 创建Nature风格预览图像，使用兰伯特投影，分辨率: {resolution_km}km")

    # 加载边界数据
    province_no_TW_AM_HK, province_TW_AM_HK, world_boundary, nine_dash_line = load_boundary_data(base_path)

    # 创建自定义颜色映射（5级）
    cmap = ListedColormap(colors)

    # -------- 图更紧凑：宽度缩小一点 --------
    fig = plt.figure(figsize=(12, 10))  # 原来(12,10)，这里更窄更紧凑

    proj = ccrs.LambertConformal(central_longitude=105, standard_parallels=(25, 47))
    ax = plt.axes(projection=proj)

    # -------- 经度范围收窄：左边界加一点，右边界小一点 --------
    # 原来 [73, 135, 18, 54]，这里改为更紧凑且仍覆盖中国+台湾
    ax.set_extent([80, 128, 18, 54], crs=ccrs.PlateCarree())

    # 底图
    ax.add_feature(cfeature.LAND, color="#fefefe", alpha=0.8)
    ax.add_feature(cfeature.OCEAN, color="#bebebe", alpha=0.8)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5, alpha=0.6)

    # 兼容不同 cartopy 版本的 proj4 字符串
    try:
        proj4_str = proj.proj4_init
    except AttributeError:
        proj4_str = proj.as_proj4()

    # # 世界国界
    # print("添加世界国界线...")
    # world_boundary_proj = world_boundary.to_crs(proj4_str)
    # ax.add_geometries(
    #     world_boundary_proj.geometry, crs=proj,
    #     facecolor='none', edgecolor='black',
    #     linewidth=0.3, alpha=0.4, zorder=3
    # )

    # -------- 显示栅格（港澳台不在 grid_values 有效区）--------
    data_to_plot = np.ma.masked_where(grid_values == -9999, grid_values)

    x_mercator = np.linspace(bounds[0], bounds[2], data_to_plot.shape[1])
    y_mercator = np.linspace(bounds[1], bounds[3], data_to_plot.shape[0])
    xx_mercator, yy_mercator = np.meshgrid(x_mercator, y_mercator)

    transformer_to_lambert = Transformer.from_crs("EPSG:3857", proj4_str, always_xy=True)
    xx_lambert, yy_lambert = transformer_to_lambert.transform(xx_mercator, yy_mercator)

    x_min_lambert, y_min_lambert = transformer_to_lambert.transform(bounds[0], bounds[1])
    x_max_lambert, y_max_lambert = transformer_to_lambert.transform(bounds[2], bounds[3])

    x_lambert_regular = np.linspace(x_min_lambert, x_max_lambert, data_to_plot.shape[1])
    y_lambert_regular = np.linspace(y_min_lambert, y_max_lambert, data_to_plot.shape[0])

    points_lambert = np.array([xx_lambert.ravel(), yy_lambert.ravel()]).T
    values_lambert = data_to_plot.ravel()

    xx_lambert_reg, yy_lambert_reg = np.meshgrid(x_lambert_regular, y_lambert_regular)

    data_lambert_reg = griddata(
        points_lambert, values_lambert,
        (xx_lambert_reg, yy_lambert_reg),
        method='nearest'
    )
    data_lambert_reg = np.ma.masked_where(data_lambert_reg == -9999, data_lambert_reg)

    extent = [x_lambert_regular.min(), x_lambert_regular.max(),
              y_lambert_regular.min(), y_lambert_regular.max()]

    im = ax.imshow(
        data_lambert_reg,
        cmap=cmap, vmin=0, vmax=4,
        extent=extent, origin='lower',
        alpha=0.8, transform=proj,
        interpolation='nearest',
        zorder=2
    )

    # -------- 省界：不含港澳台（黑边界）--------
    print("添加中国省级边界线（不含港澳台）...")
    province_no_proj = province_no_TW_AM_HK.to_crs(proj4_str)
    ax.add_geometries(
        province_no_proj.geometry, crs=proj,
        facecolor='none', edgecolor='black',
        linewidth=0.25, alpha=0.8,
        zorder=4
    )

    # -------- 港澳台：绘图显示，但不参与分析（白色填充+黑边界）--------
    print("绘制港澳台（白色填充 + 黑色边界）...")
    province_tw_proj = province_TW_AM_HK.to_crs(proj4_str)
    ax.add_geometries(
        province_tw_proj.geometry, crs=proj,
        facecolor='white', edgecolor='black',
        linewidth=0.25, alpha=1.0,
        zorder=5
    )

    # -------- 中国国界（外轮廓）：由省界合并得到，只画外边界 --------
    china_outline_geom = province_no_proj.unary_union.union(province_tw_proj.unary_union)
    ax.add_geometries(
        [china_outline_geom], crs=proj,
        facecolor='none', edgecolor='black',
        linewidth=0.8, alpha=1.0,
        zorder=6
    )


    # -------- 九段线 --------
    print("添加九段线...")
    nine_dash_line_proj = nine_dash_line.to_crs(proj4_str)
    ax.add_geometries(
        nine_dash_line_proj.geometry, crs=proj,
        facecolor='none', edgecolor='black',
        linewidth=0.6, alpha=0.9,
        zorder=6
    )

    # 网格线与标签
    lon_ticks = [80, 90, 100, 110, 120, 130]
    lat_ticks = [20, 30, 40, 50]

    gl = ax.gridlines(
        draw_labels=True, dms=True,
        x_inline=False, y_inline=False,
        linestyle='--', alpha=0.3, color='gray',
        rotate_labels=False
    )
    gl.xlocator = plt.FixedLocator(lon_ticks)
    gl.ylocator = plt.FixedLocator(lat_ticks)
    gl.top_labels = False
    gl.right_labels = False
    gl.left_labels = True
    gl.bottom_labels = True
    gl.xlabel_style = {'size': 23, 'family': 'Times New Roman', 'rotation': 0}#经度坐标
    gl.ylabel_style = {'size': 23, 'family': 'Times New Roman', 'rotation': 0}#纬度坐标
    gl.xlines = False
    gl.ylines = False
    gl.xformatter = LongitudeFormatter()
    gl.yformatter = LatitudeFormatter()

    # -------- 图例：No data 放第一个；所有色块加黑边 lw=1 --------
    legend_elements = []
    legend_elements.append(
        plt.Rectangle((0, 0), 1, 1,
                    facecolor="white", edgecolor="black", linewidth=1,
                    label="No data")
    )
    for i in range(5):
        legend_elements.append(
            plt.Rectangle((0, 0), 1, 1,
                        facecolor=colors[i], edgecolor="black", linewidth=1,
                        label=class_names[i])
        )

    legend = ax.legend(
        handles=legend_elements,
        loc='lower left',
        bbox_to_anchor=(-0.02, -0.03),
        frameon=True,
        fancybox=False,
        framealpha=0,
        edgecolor='none',
        ncol=2,
        fontsize=21,

        # 让图例色块“宽:高 = 2:1”
        handlelength=2,   # 宽
        handleheight=1,   # 高

        # ✅ 缩短两列之间的距离（关键参数）
        columnspacing=0.8,   # 试试 0.5 ~ 1.0

        # （可选）如果你还想顺便缩短色块和文字之间的距离：
        # handletextpad=0.6,

        # （可选）行与行之间更紧凑：
        # labelspacing=0.4,
    )

    # 标题：fontsize=20 且加粗
    legend.set_title(
        'Susceptibility Level',
        prop={'family': 'Times New Roman', 'size': 23, 'weight': 'bold'}
    )

    # 图例背景透明（保持你原来的风格）
    legend.get_frame().set_facecolor('none')
    legend.get_frame().set_edgecolor('none')
    legend.get_title().set_fontfamily('Times New Roman')

    # 比例尺（沿用你的函数）
    add_scalebar(ax, 70, 49, 1000, proj)

    # 保存
    formats = {
        'png': {'dpi': 600, 'format': 'png'},
        # 'tiff': {'dpi': 600, 'format': 'tiff'}
    }
    for fmt, save_kwargs in formats.items():
        output_path = f"{output_base_path}.{fmt}"
        plt.savefig(output_path, bbox_inches='tight', facecolor='white', edgecolor='none', **save_kwargs)
        print(f"{fmt.upper()}文件已保存: {output_path}")

    plt.show()
    return fig, ax


# 主程序
def main():
    # 设置分辨率
    resolution_km = 10
    
    # 数据路径设置
    boundary_shp = os.path.join(base_path, "data", "Administrative_divisions_of_china","no_TW_AM_HK", "china_pro2_no_TW_AM_HK.shp")
    output_tiff_path = os.path.join(out_dir, f"tiff_{path_name}_{ssp}_{ssp_time}.tif")
    output_base_path = os.path.join(out_dir, f"visualization_lambert_{path_name}_{ssp}_{ssp_time}")
    
    try:
        # 创建栅格图
        grid_values, xx, yy, bounds, transform = create_susceptibility_raster(
            pre_df, boundary_shp, output_tiff_path, resolution_km
        )
        
        # 创建兰伯特投影预览图（添加base_path参数）
        create_preview_map(grid_values, bounds, output_base_path, resolution_km, base_path)
        
        print("处理完成！")
        print(f"输出文件 (分辨率: {resolution_km}km):")
        print(f"- TIFF栅格文件: {output_tiff_path}")
        print(f"- PNG预览: {output_base_path}.png")
        print(f"- TIFF预览: {output_base_path}.tiff")
        
        # 统计信息
        valid_data = grid_values[grid_values != -9999]
        print(f"\n统计信息:")
        print(f"总网格数: {grid_values.size}")
        print(f"有效网格数: {len(valid_data)}")
        print(f"网格分辨率: {resolution_km}km")
        print(f"各类别分布:")
        for i in range(5):
            count = np.sum(valid_data == i)
            percentage = count / len(valid_data) * 100
            print(f"  {class_names[i]}: {count}个网格 ({percentage:.1f}%)")
            
    except Exception as e:
        print(f"处理过程中出现错误: {str(e)}")
        import traceback
        traceback.print_exc()

# 执行主程序
if __name__ == "__main__":
    main()

南海诸岛

In [ ]:
import os
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from scipy.interpolate import griddata
from rasterio.transform import from_origin
from shapely.geometry import Point
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from pyproj import Transformer

# 设置 Times New Roman 字体
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["font.size"] = 18
plt.rcParams["svg.fonttype"] = "none"

COLORS_BY_SSP = {
    "hist": ["#ffffff", "#ddd7da", "#c3b4ba", "#a78f98", "#8b6a76"],
    "ssp1": ["#ffffff", "#d8ddd6", "#bcc6b6", "#9eac97", "#7e9278"],
    "ssp2": ["#ffffff", "#d6dcde", "#b9c4c7", "#9dabb0", "#7f9097"],
    "ssp3": ["#ffffff", "#ddd7d0", "#c5b8ab", "#ac9885", "#927a66"],
    "ssp4": ["#ffffff", "#dddccf", "#c4bda8", "#aaa186", "#8f856a"],
    "ssp5": ["#ffffff", "#e0d2ce", "#c8afa8", "#ae8a81", "#946760"],
}

# 你需要在外部提前定义：ssp, pre_df, base_path, out_dir
try:
    colors = COLORS_BY_SSP[ssp]
except KeyError:
    raise ValueError(f"未知的 ssp: {ssp}，可选值：{list(COLORS_BY_SSP.keys())}")

print("ssp =", ssp)
print("colors =", colors)

# 定义类别名称
class_names = {0: "Lowest", 1: "Low", 2: "Middle", 3: "High", 4: "Highest"}


def load_boundary_data(base_path: str):
    """加载边界数据"""
    province_shp = os.path.join(base_path, "data", "china_SHP", "省界_Project.shp")
    province_boundary = gpd.read_file(province_shp)

    world_shp = os.path.join(base_path, "data", "world_SHP", "World_countries.shp")
    world_boundary = gpd.read_file(world_shp)

    return province_boundary, world_boundary


def create_susceptibility_raster(pre_df: pd.DataFrame, boundary_shp: str, resolution_km: float):
    """
    创建地质灾害易发性分布图（不保存TIFF）
    返回：grid_values, xx, yy, bounds, transform
    """
    print(f"开始创建易发性分布图，分辨率: {resolution_km}km...")

    # 1. 读取边界文件
    print("步骤1: 读取边界文件")
    china_boundary = gpd.read_file(boundary_shp).to_crs("EPSG:3857")
    print(f"边界文件坐标系: {china_boundary.crs}")

    # 2. 获取边界范围并扩展
    print("步骤2: 计算边界范围")
    bounds = china_boundary.total_bounds
    print(f"原始边界范围: {bounds}")

    expansion_factor_degree = 5
    expansion_m = expansion_factor_degree * 111000  # 1度≈111km（粗略）
    bounds[0] -= expansion_m
    bounds[1] -= expansion_m
    bounds[2] += expansion_m
    bounds[3] += expansion_m
    print(f"扩展后边界范围: {bounds}")

    # 3. 创建网格
    print("步骤3: 创建网格")
    resolution_m = resolution_km * 1000.0
    x_min, y_min, x_max, y_max = bounds
    width = x_max - x_min
    height = y_max - y_min

    x_res = int(np.round(width / resolution_m))
    y_res = int(np.round(height / resolution_m))

    x_max_adj = x_min + x_res * resolution_m
    y_max_adj = y_min + y_res * resolution_m

    x = np.linspace(x_min, x_max_adj, x_res)
    y = np.linspace(y_min, y_max_adj, y_res)
    xx, yy = np.meshgrid(x, y)
    print(f"网格尺寸: {xx.shape}, 分辨率: {resolution_km}km")

    # 4. 准备插值数据
    print("步骤4: 准备插值数据")
    geometry = [Point(lon, lat) for lon, lat in zip(pre_df["Longitude"], pre_df["Latitude"])]
    gdf_points = gpd.GeoDataFrame(pre_df, geometry=geometry, crs="EPSG:4326").to_crs("EPSG:3857")
    points_coords = np.array([(pt.x, pt.y) for pt in gdf_points.geometry])
    susceptibility_values = pre_df["Susceptibility_Class"].values
    print(f"插值点数量: {len(points_coords)}")

    # 5. 最近邻插值
    print("步骤5: 执行最近邻插值")
    grid_values = griddata(points_coords, susceptibility_values, (xx, yy), method="nearest", fill_value=-9999)

    # 6. 创建边界掩膜
    print("步骤6: 创建边界掩膜")
    grid_points = [Point(xi, yi) for xi, yi in zip(xx.ravel(), yy.ravel())]
    grid_gdf = gpd.GeoDataFrame(geometry=grid_points, crs="EPSG:3857")
    points_in_china = gpd.sjoin(grid_gdf, china_boundary, how="inner", predicate="within")

    mask = np.zeros(xx.shape, dtype=bool)
    mask.ravel()[points_in_china.index] = True
    grid_values[~mask] = -9999
    print(f"有效网格点数量: {np.sum(mask)}")

    # 7. 取消保存TIFF文件
    print("步骤7: 跳过保存TIFF文件（已取消写出tif）")
    transform = from_origin(x_min, y_max_adj, resolution_m, resolution_m)

    return grid_values, xx, yy, (x_min, y_min, x_max_adj, y_max_adj), transform


def create_nanhai_map(grid_values, bounds, output_base_path, resolution_km, base_path):
    """
    创建南海诸岛小地图（PNG不透明：transparent=False）
    """
    print(f"创建南海诸岛小地图，分辨率: {resolution_km}km")

    province_boundary, world_boundary = load_boundary_data(base_path)

    nine_dash_line_path = os.path.join(base_path, "data", "china_SHP", "nanhai", "九段线.shp")
    nine_dash_line = gpd.read_file(nine_dash_line_path)

    cmap = ListedColormap(colors)

    fig = plt.figure(figsize=(8, 6))
    proj = ccrs.PlateCarree()
    ax = plt.axes(projection=proj)

    # 设置南海诸岛范围
    ax.set_extent([105, 120, 2, 24], crs=proj)

    # 添加海洋和陆地（按你的配色，其它要素不动）
    ax.add_feature(cfeature.LAND, color="#F7F7F7", alpha=0.8)
    ax.add_feature(cfeature.OCEAN, color="#CECECE", alpha=0.8)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5, alpha=0.8)

    # 添加世界国界线（保持原样）
    world_boundary_proj = world_boundary.to_crs(proj.proj4_init)
    ax.add_geometries(
        world_boundary_proj.geometry,
        crs=proj,
        facecolor="none",
        edgecolor="gray",
        linewidth=0.3,
        alpha=0.6,
    )

    # 添加中国省级边界线（保持原样）
    province_boundary_proj = province_boundary.to_crs(proj.proj4_init)
    ax.add_geometries(
        province_boundary_proj.geometry,
        crs=proj,
        facecolor="none",
        edgecolor="black",
        linewidth=0.5,
        alpha=0.8,
    )

    # 添加九段线（保持原样）
    nine_dash_line_proj = nine_dash_line.to_crs(proj.proj4_init)
    ax.add_geometries(
        nine_dash_line_proj.geometry,
        crs=proj,
        facecolor="none",
        edgecolor="black",
        linewidth=1.5,
        alpha=1.0,
        linestyle="--",
    )

    # 创建用于显示的网格
    data_to_plot = np.ma.masked_where(grid_values == -9999, grid_values)

    # Web Mercator 坐标网格
    x_mercator = np.linspace(bounds[0], bounds[2], data_to_plot.shape[1])
    y_mercator = np.linspace(bounds[1], bounds[3], data_to_plot.shape[0])
    xx_mercator, yy_mercator = np.meshgrid(x_mercator, y_mercator)

    # 转 WGS84（经纬度）用于 PlateCarree 叠加显示
    transformer_to_wgs84 = Transformer.from_crs("EPSG:3857", "EPSG:4326", always_xy=True)
    xx_wgs84, yy_wgs84 = transformer_to_wgs84.transform(xx_mercator, yy_mercator)

    # 绘制栅格
    ax.pcolormesh(
        xx_wgs84,
        yy_wgs84,
        data_to_plot,
        cmap=cmap,
        vmin=0,
        vmax=4,
        alpha=0.7,
        transform=proj,
    )

    # 移除坐标轴
    ax.set_xticks([])
    ax.set_yticks([])
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["bottom"].set_visible(False)
    ax.spines["left"].set_visible(False)

    # 仅修改图的外边框颜色为白色（不影响省界/国界/海岸线等）
    if hasattr(ax, "outline_patch") and ax.outline_patch is not None:
        ax.outline_patch.set_edgecolor("white")
        ax.outline_patch.set_linewidth(1.2)
        ax.outline_patch.set_zorder(10)
    if "geo" in ax.spines:
        ax.spines["geo"].set_edgecolor("white")
        ax.spines["geo"].set_linewidth(1.2)
        ax.spines["geo"].set_zorder(10)

    # 让导出PNG完全不透明（关键：transparent=False）
    fig.patch.set_facecolor("white")
    ax.set_facecolor("white")

    # 保存图像（统一输出：*_南海.png）
    os.makedirs(os.path.dirname(output_base_path), exist_ok=True)
    output_prefix = f"{output_base_path}"
    output_path = f"{output_prefix}.png"

    plt.savefig(
        output_path,
        bbox_inches="tight",
        pad_inches=0,
        facecolor="white",
        edgecolor="white",
        transparent=False,  # 关键：PNG不透明（透明度=0的常用表达）
        dpi=600,
        format="png",
    )
    print(f"PNG文件已保存: {output_path}")

    plt.show()
    return fig, ax


def main():
    resolution_km = 10

    boundary_shp = os.path.join(base_path, "data", "china_SHP", "国界_Project.shp")
    output_base_path = os.path.join(out_dir, f"susceptibility_map_nanhai_{resolution_km}km")

    os.makedirs(out_dir, exist_ok=True)

    try:
        # 创建栅格（不保存tif）
        grid_values, xx, yy, bounds, transform = create_susceptibility_raster(
            pre_df, boundary_shp, resolution_km
        )

        # 创建南海诸岛小地图（PNG不透明）
        create_nanhai_map(grid_values, bounds, output_base_path, resolution_km, base_path)

        print("处理完成！")
        print(f"输出文件 (分辨率: {resolution_km}km):")
        print(f"- PNG预览: {output_base_path}.png")

        # 统计信息
        valid_data = grid_values[grid_values != -9999]
        print("\n统计信息:")
        print(f"总网格数: {grid_values.size}")
        print(f"有效网格数: {len(valid_data)}")
        print(f"网格分辨率: {resolution_km}km")
        print("各类别分布:")
        for i in range(5):
            count = np.sum(valid_data == i)
            percentage = count / len(valid_data) * 100
            print(f"  {class_names[i]}: {count}个网格 ({percentage:.1f}%)")

    except Exception as e:
        print(f"处理过程中出现错误: {str(e)}")
        import traceback
        traceback.print_exc()


if __name__ == "__main__":
    main()


按省统计5类风险区的面积和所占比例

In [ ]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from scipy.interpolate import griddata

import rasterio
from rasterio.transform import from_origin
from rasterio.features import geometry_mask, rasterize

# =========================
# 0) 等级定义（0-4）
# =========================
# 0=Lowest, 1=Low, 2=Middle, 3=High, 4=Highest
class_names = {0: 'Lowest', 1: 'Low', 2: 'Middle', 3: 'High', 4: 'Highest'}

# Shapefile 字段名 <=10 字符
count_fields = {
    0: "Lowest_N",
    1: "Low_N",
    2: "Middle_N",
    3: "High_N",
    4: "Highest_N"
}
prop_fields = {
    0: "Lowest_P",
    1: "Low_P",
    2: "Middle_P",
    3: "High_P",
    4: "Highest_P"
}

# =========================
# 1) 创建分类栅格（GeoTIFF, 0-4 + NoData）
# =========================
def create_susceptibility_raster(
    pre_df: pd.DataFrame,
    boundary_shp: str,
    output_tif_path: str,
    resolution_km: float,
    expand_degree: float = 5.0,
    nodata_value: int = 255
):
    """
    输出：ArcGIS Pro 可直接导入的 GeoTIFF 分类栅格（0-4），NoData=255
    栅格 CRS：EPSG:3857
    """
    print(f"开始创建分类栅格，分辨率: {resolution_km} km")

    china_boundary = gpd.read_file(boundary_shp)
    if china_boundary.crs is None:
        raise ValueError("boundary_shp 没有 CRS，请先定义坐标系。")
    china_boundary_3857 = china_boundary.to_crs("EPSG:3857")

    bounds = china_boundary_3857.total_bounds
    expansion_m = expand_degree * 111000

    x_min = bounds[0] - expansion_m
    y_min = bounds[1] - expansion_m
    x_max = bounds[2] + expansion_m
    y_max = bounds[3] + expansion_m

    resolution_m = resolution_km * 1000.0
    width = x_max - x_min
    height = y_max - y_min
    n_cols = int(np.ceil(width / resolution_m))
    n_rows = int(np.ceil(height / resolution_m))

    x_max_adj = x_min + n_cols * resolution_m
    y_max_adj = y_min + n_rows * resolution_m

    transform = from_origin(x_min, y_max_adj, resolution_m, resolution_m)

    print(f"栅格大小 rows x cols = {n_rows} x {n_cols}")
    print("输出 CRS: EPSG:3857")
    print(f"NoData: {nodata_value} (有效类别: 0-4)")

    x_centers = x_min + (np.arange(n_cols) + 0.5) * resolution_m
    y_centers = y_max_adj - (np.arange(n_rows) + 0.5) * resolution_m
    xx, yy = np.meshgrid(x_centers, y_centers)

    required_cols = {"Longitude", "Latitude", "Susceptibility_Class"}
    if not required_cols.issubset(pre_df.columns):
        raise ValueError(f"pre_df 必须包含列: {required_cols}")

    pre_df = pre_df.copy()
    pre_df["Susceptibility_Class"] = pre_df["Susceptibility_Class"].astype(int)
    if pre_df["Susceptibility_Class"].min() < 0 or pre_df["Susceptibility_Class"].max() > 4:
        raise ValueError("Susceptibility_Class 必须是 0-4 五类。")

    gdf_points = gpd.GeoDataFrame(
        pre_df,
        geometry=[Point(lon, lat) for lon, lat in zip(pre_df["Longitude"], pre_df["Latitude"])],
        crs="EPSG:4326"
    ).to_crs("EPSG:3857")

    pts = np.column_stack([gdf_points.geometry.x.values, gdf_points.geometry.y.values])
    vals = pre_df["Susceptibility_Class"].values
    print(f"插值点数量: {len(pts)}")

    grid_values = griddata(pts, vals, (xx, yy), method="nearest").astype(np.uint8)

    mask_in = geometry_mask(
        geometries=china_boundary_3857.geometry,
        transform=transform,
        invert=True,
        out_shape=(n_rows, n_cols)
    )

    out = np.full((n_rows, n_cols), nodata_value, dtype=np.uint8)
    out[mask_in] = grid_values[mask_in]

    os.makedirs(os.path.dirname(output_tif_path), exist_ok=True)
    with rasterio.open(
        output_tif_path,
        "w",
        driver="GTiff",
        height=n_rows,
        width=n_cols,
        count=1,
        dtype="uint8",
        crs="EPSG:3857",
        transform=transform,
        nodata=nodata_value,
        compress="LZW"
    ) as dst:
        dst.write(out, 1)

    if not os.path.exists(output_tif_path):
        raise RuntimeError(f"GeoTIFF 写入失败（路径不存在）: {output_tif_path}")

    print(f"GeoTIFF 已保存: {output_tif_path}")
    return out, transform, "EPSG:3857", nodata_value


# =========================
# 2) 省级统计（输出省面 shp + 5类像元数量 + 5类比例）
# =========================
def province_zonal_counts_to_shp(
    province_shp: str,
    raster_array: np.ndarray,
    transform,
    raster_crs: str,
    nodata_value: int,
    out_province_shp: str
):
    """
    输出字段：
    - NAME, ADCODE99
    - Lowest_N, Low_N, Middle_N, High_N, Highest_N
    - Lowest_P, Low_P, Middle_P, High_P, Highest_P  (0-1比例)
    """
    prov_raw = gpd.read_file(province_shp)
    if prov_raw.crs is None:
        raise ValueError("province_shp 没有 CRS，请先定义坐标系。")

    need = {"NAME", "ADCODE99"}
    miss = need - set(prov_raw.columns)
    if miss:
        raise ValueError(f"province_shp 缺少字段: {miss}，请检查字段名。")

    prov_raw = prov_raw.to_crs(raster_crs)
    prov_raw = prov_raw[["NAME", "ADCODE99", "geometry"]].copy()

    # dissolve：按 ADCODE99 合并成省级
    prov = prov_raw.dissolve(by="ADCODE99", as_index=False)

    # NAME 修复：取每个 ADCODE99 的第一个 NAME
    name_map = prov_raw.dropna(subset=["ADCODE99", "NAME"]).groupby("ADCODE99")["NAME"].first().to_dict()
    prov["NAME"] = prov["ADCODE99"].map(name_map)

    prov = prov[prov.geometry.notna()].copy()
    prov["geometry"] = prov["geometry"].buffer(0)

    print(f"省级行政区面数量（dissolve后）: {len(prov)}  (期望约 34)")

    n_rows, n_cols = raster_array.shape

    prov = prov.reset_index(drop=True).copy()
    prov["zone_id"] = (prov.index.astype(np.int32) + 1)

    zone_raster = rasterize(
        [(geom, zid) for geom, zid in zip(prov.geometry, prov["zone_id"])],
        out_shape=(n_rows, n_cols),
        transform=transform,
        fill=0,
        dtype="int32",
        all_touched=False
    )

    valid_mask = (raster_array != nodata_value) & (zone_raster > 0)
    zones = zone_raster[valid_mask]
    classes = raster_array[valid_mask].astype(int)

    # 初始化 count / prop 列
    for k in range(5):
        prov[count_fields[k]] = 0
        prov[prop_fields[k]] = 0.0

    # 统计（按 zone_id）
    order = np.argsort(zones)
    zones_sorted = zones[order]
    classes_sorted = classes[order]

    unique_zones, start_idx = np.unique(zones_sorted, return_index=True)
    end_idx = np.r_[start_idx[1:], len(zones_sorted)]

    for zid, s, e in zip(unique_zones, start_idx, end_idx):
        hist = np.bincount(classes_sorted[s:e], minlength=5)  # count0..4
        total = int(hist.sum())
        row = int(zid - 1)

        # 写 count
        for k in range(5):
            prov.at[row, count_fields[k]] = int(hist[k])

        # 写 proportion（0-1），避免除零
        if total > 0:
            for k in range(5):
                prov.at[row, prop_fields[k]] = float(hist[k] / total)
                # 如果你想存百分数（0-100），改成：
                # prov.at[row, prop_fields[k]] = float(hist[k] / total * 100.0)

    out = prov[
        ["NAME", "ADCODE99"]
        + [count_fields[i] for i in range(5)]
        + [prop_fields[i] for i in range(5)]
        + ["geometry"]
    ].copy()

    os.makedirs(os.path.dirname(out_province_shp), exist_ok=True)
    out.to_file(out_province_shp, driver="ESRI Shapefile", encoding="utf-8")

    if not os.path.exists(out_province_shp):
        raise RuntimeError(f"省级 SHP 写入失败（路径不存在）: {out_province_shp}")

    print(f"省级统计 SHP 已保存: {out_province_shp}")
    return out


# =========================
# 3) 主入口：使用已有 pre_df
# =========================
def run(pre_df, base_path, path_name, resolution_km=10):
    boundary_shp = os.path.join(base_path, "data", "china_SHP", "国界_Project.shp")

    # ✅ 你的省级行政区 shp（绝对路径）
    province_shp = "/path/to/sinkhole-risk-china/data/Administrative_divisions_of_china/china_pro2.shp"

    # ✅ 文件名：xxxx_path_name.xxx（去掉 resolution_km）
    out_raster_tif = os.path.join(out_dir, f"susceptibility_{path_name}_{ssp}_{ssp_time}.tif")
    out_prov_shp   = os.path.join(out_dir, f"province_counts_{path_name}_{ssp}_{ssp_time}.shp")

    raster_array, transform, raster_crs, nodata = create_susceptibility_raster(
        pre_df=pre_df,
        boundary_shp=boundary_shp,
        output_tif_path=out_raster_tif,
        resolution_km=resolution_km,
        expand_degree=5.0,
        nodata_value=255
    )

    province_zonal_counts_to_shp(
        province_shp=province_shp,
        raster_array=raster_array,
        transform=transform,
        raster_crs=raster_crs,
        nodata_value=nodata,
        out_province_shp=out_prov_shp
    )

    print("\n完成！输出：")
    print(f"- 栅格 GeoTIFF: {out_raster_tif}")
    print(f"- 省级统计 SHP: {out_prov_shp}")


# 用法示例（你外部已定义 pre_df、base_path）：
run(pre_df, base_path, path_name, resolution_km=10)


统计各省（除了港澳台）的风险水平相较于2020年的如何变化。

In [ ]:
# ============================================
# 省级统计：当前(ssp/ssp_time)相对 2020(hist) 的原始 GWR 风险值变化（经纬度对齐 + shp补ADCODE99）
# 不再使用 sigmoid 压缩概率，而是直接用原始 GWR 回归值计算变化率；按变化率降序排列
# 输出：CSV 保存到 out_dir（文件名包含 ssp 与 ssp_time + 关键词）
# ============================================
import os
import glob
import numpy as np
import pandas as pd

# ---- 0) 基本检查 ----
required_vars = ["pre_df", "y_pred_gwr", "y_pred_gwr_hist_2020", "out_dir", "base_path", "data_base_path", "resolution", "ssp", "ssp_time"]
missing_vars = [v for v in required_vars if v not in globals()]
if missing_vars:
    raise NameError(f"缺少变量：{missing_vars}。请先运行前面的读取/预测结果代码块。")

# ---- 1) 统一经纬度列名（只用于匹配/空间连接）----
def pick_lon_lat_cols(df: pd.DataFrame):
    lon_candidates = ["Longitude", "longitude", "LON", "lon", "X", "x"]
    lat_candidates = ["Latitude", "latitude", "LAT", "lat", "Y", "y"]
    lon_col = next((c for c in lon_candidates if c in df.columns), None)
    lat_col = next((c for c in lat_candidates if c in df.columns), None)
    if lon_col is None or lat_col is None:
        raise KeyError(f"无法在 df 中找到经纬度列。可选列名示例：{lon_candidates} / {lat_candidates}")
    return lon_col, lat_col

# 当前点 + 当前原始 GWR 值
lon_col_cur, lat_col_cur = pick_lon_lat_cols(pre_df)
df_cur = pre_df[[lon_col_cur, lat_col_cur]].copy()
df_cur.rename(columns={lon_col_cur: "Longitude", lat_col_cur: "Latitude"}, inplace=True)

y_cur = np.asarray(y_pred_gwr, dtype=float).reshape(-1)
if len(df_cur) != len(y_cur):
    raise ValueError(
        "当前点数据行数与 y_pred_gwr 长度不一致：\n"
        f"len(df_cur)={len(df_cur)}, len(y_cur)={len(y_cur)}"
    )
df_cur["GWR_Score_Current"] = y_cur

# ---- 2) 自动找到 hist/2020 的点 CSV 并读入（只需要经纬度）----
def find_hist_2020_csv(data_base_path: str, resolution: str) -> str:
    patterns = [
        os.path.join(data_base_path, "**", f"*Points_China_all_{resolution}*hist*2020*.csv"),
        os.path.join(data_base_path, "**", f"*Points_China_all_{resolution}*hist*.csv"),
    ]
    cands = []
    for p in patterns:
        cands.extend(glob.glob(p, recursive=True))
    cands = sorted(set(cands))
    if not cands:
        raise FileNotFoundError("在 data_base_path 下没有找到 hist/2020 的点 CSV（文件名需包含 hist 和/或 2020）。")

    def score(fp: str) -> tuple:
        name = os.path.basename(fp).lower()
        s = 0
        if "hist" in name:
            s += 10
        if "2020" in name:
            s += 10
        if "cleaned" in name:
            s += 2
        if f"points_china_all_{resolution}".lower() in name:
            s += 5
        return (s, os.path.getmtime(fp))

    return sorted(cands, key=score, reverse=True)[0]

hist_csv = find_hist_2020_csv(data_base_path, resolution)
print(f"✅ 自动匹配到 2020(hist) 点文件：{hist_csv}")

df_2020_raw = pd.read_csv(hist_csv)
lon_col_2020, lat_col_2020 = pick_lon_lat_cols(df_2020_raw)

df_2020 = df_2020_raw[[lon_col_2020, lat_col_2020]].copy()
df_2020.rename(columns={lon_col_2020: "Longitude", lat_col_2020: "Latitude"}, inplace=True)

y_2020 = np.asarray(y_pred_gwr_hist_2020, dtype=float).reshape(-1)
if len(df_2020) != len(y_2020):
    raise ValueError(
        "2020 点CSV行数与 y_pred_gwr_hist_2020 长度不一致：\n"
        f"len(df_2020)={len(df_2020)}, len(y_2020)={len(y_2020)}\n"
        "请确保该CSV对应生成该pkl的点文件。"
    )
df_2020["GWR_Score_2020"] = y_2020

# ---- 3) 原始 GWR 值数值清洗 ----
df_cur["GWR_Score_Current"] = pd.to_numeric(df_cur["GWR_Score_Current"], errors="coerce")
df_2020["GWR_Score_2020"] = pd.to_numeric(df_2020["GWR_Score_2020"], errors="coerce")

# ---- 4) 仅用经纬度对齐：round 后按 (lon_r, lat_r) merge ----
def add_xy_key(df: pd.DataFrame, nd: int = 6) -> pd.DataFrame:
    out = df.copy()
    out["lon_r"] = pd.to_numeric(out["Longitude"], errors="coerce").round(nd)
    out["lat_r"] = pd.to_numeric(out["Latitude"], errors="coerce").round(nd)
    out = out.dropna(subset=["lon_r", "lat_r"])
    return out

df_cur_k = add_xy_key(df_cur, nd=6)
df_2020_k = add_xy_key(df_2020, nd=6)

# 如果同一坐标重复：对该坐标的原始 GWR 值取均值，避免 merge 一对多
cur_pts = df_cur_k.groupby(["lon_r", "lat_r"], as_index=False).agg(
    GWR_Score_Current=("GWR_Score_Current", "mean")
)
hist_pts = df_2020_k.groupby(["lon_r", "lat_r"], as_index=False).agg(
    GWR_Score_2020=("GWR_Score_2020", "mean")
)

aligned = cur_pts.merge(hist_pts, on=["lon_r", "lat_r"], how="inner")

print(f"当前点数(去重后)：{len(cur_pts)}")
print(f"2020点数(去重后)：{len(hist_pts)}")
print(f"成功对齐点数：{len(aligned)}")
print(f"对齐率(相对当前)：{len(aligned)/len(cur_pts):.2%}；(相对2020)：{len(aligned)/len(hist_pts):.2%}")

if len(aligned) == 0:
    raise ValueError("对齐后为 0 个点：请检查两套点的坐标是否一致（round 位数、坐标系或点集是否相同）。")

# ---- 5) 对齐后用 shp 加省信息：只取 ADCODE99 ----
import geopandas as gpd

province_no_TW_AM_HK_shp = os.path.join(
    base_path, "data", "Administrative_divisions_of_china", "no_TW_AM_HK", "china_pro2_no_TW_AM_HK.shp"
)

prov_raw = gpd.read_file(province_no_TW_AM_HK_shp)

adcode_col = None
for c in prov_raw.columns:
    if str(c).strip().lower() == "adcode99":
        adcode_col = c
        break
if adcode_col is None:
    raise KeyError(f"shp 中找不到 ADCODE99 列。当前列名为：{list(prov_raw.columns)}")

prov = prov_raw[[adcode_col, "geometry"]].copy().rename(columns={adcode_col: "ADCODE99"})

if prov.crs is None:
    prov = prov.set_crs("EPSG:4326")

g_aligned = gpd.GeoDataFrame(
    aligned.copy(),
    geometry=gpd.points_from_xy(aligned["lon_r"], aligned["lat_r"]),
    crs="EPSG:4326"
).to_crs(prov.crs)

joined = gpd.sjoin(g_aligned, prov, how="left", predicate="within")
joined = joined.dropna(subset=["ADCODE99"]).copy()
joined["ADCODE99"] = pd.to_numeric(joined["ADCODE99"], errors="coerce").astype("Int64")

# ---- 6) 用你给的映射表补 NAME_EN_JX ----
province_name_map = {
    510000: "Sichuan",
    460000: "Hainan",
    410000: "Henan",
    310000: "Shanghai",
    120000: "Tianjin",
    110000: "Beijing",
    540000: "Tibet",
    630000: "Qinghai",
    640000: "Ningxia",
    350000: "Fujian",
    620000: "Gansu",
    530000: "Yunnan",
    320000: "Jiangsu",
    330000: "Zhejiang",
    230000: "Heilongjiang",
    220000: "Jilin",
    150000: "Inner Mongolia",
    610000: "Shaanxi",
    340000: "Anhui",
    500000: "Chongqing",
    650000: "Xinjiang",
    130000: "Hebei",
    420000: "Hubei",
    440000: "Guangdong",
    210000: "Liaoning",
    370000: "Shandong",
    520000: "Guizhou",
    360000: "Jiangxi",
    430000: "Hunan",
    450000: "Guangxi",
    140000: "Shanxi",
}
joined["NAME_EN_JX"] = joined["ADCODE99"].map(province_name_map)
joined = joined.dropna(subset=["NAME_EN_JX"]).copy()

# ---- 7) 省级统计变化 + 变化率 ----
stats = (
    joined.groupby(["ADCODE99", "NAME_EN_JX"], dropna=False)
    .agg(
        n_points=("GWR_Score_Current", "size"),
        mean_score_2020=("GWR_Score_2020", "mean"),
        mean_score_current=("GWR_Score_Current", "mean"),
        median_score_2020=("GWR_Score_2020", "median"),
        median_score_current=("GWR_Score_Current", "median"),
    )
    .reset_index()
)

stats["delta_mean"] = stats["mean_score_current"] - stats["mean_score_2020"]
stats["delta_median"] = stats["median_score_current"] - stats["median_score_2020"]

# 原始 GWR 均值可能为负，分母使用 |mean_score_2020|，避免基准为负时方向失真
stats["change_rate"] = np.where(
    np.abs(stats["mean_score_2020"]) > 1e-12,
    stats["delta_mean"] / np.abs(stats["mean_score_2020"]),
    np.nan
)
stats["change_rate_pct"] = stats["change_rate"] * 100.0

# ---- 8) 全国汇总（基于成功落省的对齐点）----
china_mean_2020 = float(joined["GWR_Score_2020"].mean())
china_mean_cur = float(joined["GWR_Score_Current"].mean())
china_delta = china_mean_cur - china_mean_2020
china_base = abs(china_mean_2020)
china_change_rate = (china_delta / china_base) if china_base > 1e-12 else np.nan

china = pd.DataFrame([{
    "ADCODE99": 100000,
    "NAME_EN_JX": "China",
    "n_points": int(len(joined)),
    "mean_score_2020": china_mean_2020,
    "mean_score_current": china_mean_cur,
    "delta_mean": float(china_delta),
    "change_rate": float(china_change_rate) if not pd.isna(china_change_rate) else np.nan,
    "change_rate_pct": float(china_change_rate * 100.0) if not pd.isna(china_change_rate) else np.nan,
    "median_score_2020": float(joined["GWR_Score_2020"].median()),
    "median_score_current": float(joined["GWR_Score_Current"].median()),
    "delta_median": float(joined["GWR_Score_Current"].median() - joined["GWR_Score_2020"].median()),
}])

out = pd.concat([china, stats], ignore_index=True)

# ---- 9) 按变化率降序排序（NaN 放最后）----
out = out.sort_values(by=["change_rate"], ascending=False, na_position="last").reset_index(drop=True)

out = out[[
    "ADCODE99", "NAME_EN_JX",
    "n_points",
    "mean_score_2020", "mean_score_current", "delta_mean",
    "change_rate", "change_rate_pct",
    "median_score_2020", "median_score_current", "delta_median",
]]

# ---- 10) 保存 CSV ----
csv_name = f"province_raw_gwr_change_vs2020_lonlat_align_sortByChangeRate_{ssp}_{ssp_time}.csv"
csv_path = os.path.join(out_dir, csv_name)
out.to_csv(csv_path, index=False, encoding="utf-8-sig")

print("✅ 变化率已改为基于原始 GWR score 计算；分母使用 |mean_score_2020|。")
print(f"✅ 已保存：{csv_path}")
display(out.head(15))


棒棒糖图

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.transforms as mtransforms
from pathlib import Path
from matplotlib.colors import LinearSegmentedColormap

plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["font.size"] = 18
plt.rcParams["svg.fonttype"] = "none"


def resolve_province_change_csv() -> str:
    preferred_names = []
    if all(k in globals() for k in ["out_dir", "ssp", "ssp_time"]):
        preferred_names = [
            os.path.join(out_dir, f"province_raw_gwr_change_vs2020_lonlat_align_sortByChangeRate_{ssp}_{ssp_time}.csv"),
            os.path.join(out_dir, f"province_prob_change_vs2020_lonlat_align_clip01_sortByChangeRate_{ssp}_{ssp_time}.csv"),
        ]
    for candidate in preferred_names:
        if os.path.exists(candidate):
            return candidate

    if "csv_path" in globals() and os.path.exists(csv_path):
        return csv_path

    if "out_dir" in globals():
        patterns = [
            os.path.join(out_dir, "province_raw_gwr_change_vs2020_lonlat_align_sortByChangeRate_*.csv"),
            os.path.join(out_dir, "province_prob_change_vs2020_lonlat_align_clip01_sortByChangeRate_*.csv"),
        ]
        matches = []
        for pattern in patterns:
            matches.extend(glob.glob(pattern))
        matches = sorted(set(matches), key=os.path.getmtime, reverse=True)
        if matches:
            return matches[0]

    raise FileNotFoundError("未找到省级变化率 CSV。请先运行上一格“省级统计”代码块。")


def build_point_colors(values: np.ndarray):
    pos_cmap = LinearSegmentedColormap.from_list("rise", ["#d9d1d3", "#8b666f"])
    neg_cmap = LinearSegmentedColormap.from_list("decline", ["#d9eaee", "#77bed0"])

    values = np.asarray(values, dtype=float)
    pos_values = values[values > 0]
    neg_values = np.abs(values[values < 0])

    pos_min = pos_values.min() if pos_values.size else 0.0
    pos_max = pos_values.max() if pos_values.size else 1.0
    neg_min = neg_values.min() if neg_values.size else 0.0
    neg_max = neg_values.max() if neg_values.size else 1.0

    colors = []
    for value in values:
        if value > 0:
            if pos_max - pos_min < 1e-12:
                t = 1.0
            else:
                t = (value - pos_min) / (pos_max - pos_min)
            colors.append(pos_cmap(0.25 + 0.70 * t))
        elif value < 0:
            magnitude = abs(value)
            if neg_max - neg_min < 1e-12:
                t = 1.0
            else:
                t = (magnitude - neg_min) / (neg_max - neg_min)
            colors.append(neg_cmap(0.25 + 0.70 * t))
        else:
            colors.append("#d9dfe2")
    return colors


def draw_lollipops(ax, x, y, colors, fig, stem_width=3.6, marker_size=235):
    highlight_offset = mtransforms.ScaledTranslation(-2.2 / 72.0, 2.2 / 72.0, fig.dpi_scale_trans)
    for xi, yi, ci in zip(x, y, colors):
        ax.vlines(xi, 0, yi, color=ci, linewidth=stem_width, zorder=2, alpha=0.98)
        ax.scatter(xi, yi, s=marker_size, color=[ci], edgecolor=[ci], linewidth=0.8, zorder=4)
        ax.scatter(
            [xi], [yi], s=36, color="white", alpha=0.82, edgecolor="none",
            transform=ax.transData + highlight_offset, zorder=5,
        )


def style_axis(ax):
    ax.set_axisbelow(True)
    ax.yaxis.grid(True, linestyle=(0, (5, 5)), color="#7c7c7c", linewidth=0.8, alpha=0.95)
    ax.xaxis.grid(False)
    for spine in ax.spines.values():
        spine.set_color("#8a8a8a")
        spine.set_linewidth(1.8)
    ax.tick_params(axis="x", width=1.8, length=9, color="#8a8a8a")
    ax.tick_params(axis="y", width=1.6, length=8, color="#8a8a8a", labelsize=23)


def add_break_marks(ax_top, ax_bottom, size=0.012, color="#8a8a8a"):
    kwargs = dict(color=color, clip_on=False, linewidth=1.8)
    ax_top.plot((-size, +size), (-size, +size), transform=ax_top.transAxes, **kwargs)
    ax_top.plot((1 - size, 1 + size), (-size, +size), transform=ax_top.transAxes, **kwargs)
    ax_bottom.plot((-size, +size), (1 - size, 1 + size), transform=ax_bottom.transAxes, **kwargs)
    ax_bottom.plot((1 - size, 1 + size), (1 - size, 1 + size), transform=ax_bottom.transAxes, **kwargs)


def add_direction_labels(ax):
    ax.scatter([0.035], [0.11], s=260, color="#8b666f", transform=ax.transAxes, clip_on=False, zorder=6)
    ax.text(0.07, 0.105, "Risk level rising", transform=ax.transAxes, ha="left", va="center", fontsize=24)
    ax.scatter([0.49], [0.11], s=260, color="#77bed0", transform=ax.transAxes, clip_on=False, zorder=6)
    ax.text(0.525, 0.105, "Risk level declining", transform=ax.transAxes, ha="left", va="center", fontsize=24)


change_csv_path = resolve_province_change_csv()
df_change = pd.read_csv(change_csv_path, encoding="utf-8-sig")

if "change_rate_pct" not in df_change.columns:
    if "change_rate" in df_change.columns:
        df_change["change_rate_pct"] = pd.to_numeric(df_change["change_rate"], errors="coerce") * 100.0
    else:
        raise KeyError("CSV 缺少 change_rate_pct / change_rate 列。")

plot_df = df_change[["NAME_EN_JX", "change_rate_pct"]].copy()
plot_df["change_rate_pct"] = pd.to_numeric(plot_df["change_rate_pct"], errors="coerce")
plot_df = plot_df.dropna(subset=["NAME_EN_JX", "change_rate_pct"]).copy()
plot_df = plot_df.sort_values("change_rate_pct", ascending=False, kind="mergesort").reset_index(drop=True)

if plot_df.empty:
    raise ValueError("用于绘图的省级变化率数据为空。")

x = np.arange(len(plot_df))
y = plot_df["change_rate_pct"].to_numpy(dtype=float)
labels = plot_df["NAME_EN_JX"].astype(str).tolist()
colors = build_point_colors(y)

china_idx = None
china_value = None
china_rows = plot_df.index[plot_df["NAME_EN_JX"].str.strip().eq("China")]
if len(china_rows) > 0:
    china_idx = int(china_rows[0])
    china_value = float(plot_df.loc[china_idx, "change_rate_pct"])

use_break = float(np.nanmax(y)) > 50.0

if use_break:
    fig, (ax_top, ax_bottom) = plt.subplots(
        2,
        1,
        figsize=(16.5, 6.8),
        sharex=True,
        gridspec_kw={"height_ratios": [1.0, 4.2], "hspace": 0.05},
    )
    for ax in (ax_top, ax_bottom):
        draw_lollipops(ax, x, y, colors, fig)
        style_axis(ax)

    lower_min = min(-40.0, np.floor(np.nanmin(y) / 5.0) * 5.0 - 5.0)
    lower_max = 35.0
    upper_min = 50.0
    upper_max = max(150.0, np.ceil(np.nanmax(y) / 10.0) * 10.0 + 10.0)

    ax_bottom.set_ylim(lower_min, lower_max)
    ax_top.set_ylim(upper_min, upper_max)
    ax_bottom.axhline(0, color="#0a6ea8", linewidth=3.6, zorder=3)

    bottom_ticks = [tick for tick in [-40, -20, 0, 20] if lower_min <= tick <= lower_max]
    if not bottom_ticks:
        bottom_ticks = np.linspace(lower_min, lower_max, 4)
    top_ticks = np.arange(upper_min, upper_max + 0.1, 50.0)
    ax_bottom.set_yticks(bottom_ticks)
    ax_top.set_yticks(top_ticks)

    ax_top.spines["bottom"].set_visible(False)
    ax_bottom.spines["top"].set_visible(False)
    ax_top.tick_params(axis="x", which="both", bottom=False, labelbottom=False)
    add_break_marks(ax_top, ax_bottom)

    main_ax = ax_bottom
    axes_for_highlight = [ax_top, ax_bottom]
else:
    fig, ax = plt.subplots(figsize=(16.5, 6.8))
    draw_lollipops(ax, x, y, colors, fig)
    style_axis(ax)
    max_abs = max(np.nanmax(np.abs(y)), 5.0)
    y_bound = np.ceil((max_abs * 1.18) / 5.0) * 5.0
    ax.set_ylim(-y_bound, y_bound)
    tick_step = 5.0 if y_bound <= 20 else 10.0
    ax.set_yticks(np.arange(-y_bound, y_bound + 0.1, tick_step))
    ax.axhline(0, color="#0a6ea8", linewidth=3.6, zorder=3)

    main_ax = ax
    axes_for_highlight = [ax]

for ax in axes_for_highlight:
    ax.set_xlim(-0.6, len(plot_df) - 0.4)

if china_idx is not None:
    for ax in axes_for_highlight:
        ax.axvspan(
            china_idx - 0.38,
            china_idx + 0.38,
            facecolor="none",
            edgecolor="#ec8e9c",
            linewidth=2.2,
            zorder=6,
        )

main_ax.set_xticks(x)
main_ax.set_xticklabels(labels, rotation=45, ha="right", rotation_mode="anchor", fontsize=18)
for tick_label in main_ax.get_xticklabels():
    if tick_label.get_text().strip() == "China":
        tick_label.set_color("#d96d80")
        tick_label.set_fontweight("bold")

main_ax.set_xlabel("Province", fontsize=22)
fig.text(0.032, 0.5, "Raw GWR risk change rate (%)", va="center", rotation="vertical", fontsize=28)
add_direction_labels(main_ax)

if china_idx is not None and china_value is not None:
    direction = "increased" if china_value >= 0 else "decreased"
    annotation_text = f"Nationwide risk level\nhas {direction} by {abs(china_value):.2f}%"
    x_frac = china_idx / max(len(plot_df) - 1, 1)
    if x_frac <= 0.6:
        text_x = min(x_frac + 0.035, 0.82)
        ha = "left"
    else:
        text_x = max(x_frac - 0.035, 0.22)
        ha = "right"
    text_y = 0.965 if use_break else 0.93
    main_ax.text(
        text_x,
        text_y,
        annotation_text,
        transform=main_ax.transAxes,
        ha=ha,
        va="top",
        fontsize=20,
    )

fig.subplots_adjust(left=0.09, right=0.985, bottom=0.25, top=0.96)

csv_path_obj = Path(change_csv_path)
output_stem = csv_path_obj.stem
for old_prefix in [
    "province_raw_gwr_change_vs2020_lonlat_align_sortByChangeRate",
    "province_prob_change_vs2020_lonlat_align_clip01_sortByChangeRate",
]:
    if output_stem.startswith(old_prefix):
        output_stem = output_stem.replace(old_prefix, "province_change_lollipop", 1)
        break
output_base = csv_path_obj.with_name(output_stem)

for ext, dpi in [("png", 600), ("svg", 300)]:
    out_path = f"{output_base}.{ext}"
    fig.savefig(out_path, dpi=dpi, bbox_inches="tight", facecolor="white")
    print(f"{ext.upper()} 已保存: {out_path}")

plt.show()
